# 1. The problem, and a slow first answer

**The workshop in one sentence:** take *one* operation - a 3x3 image convolution -
and make it go faster and faster, learning what the bottleneck is at each step.

### Why convolution?

A 3x3 convolution slides a small window over an image and, at every pixel, computes
nine multiply-accumulates (one per window tap). It is the inner loop of every
convolutional neural network. And it is *compute-bound*: lots of arithmetic, very
little of it reused - exactly the kind of work dedicated hardware is good at.

### How much can a processor actually do?

Order-of-magnitude, on the board's ARM core:

- A core at ~1 GHz retires roughly one instruction per clock, so ~1e9 ops/second.
- One output pixel is 9 multiplies + 8 adds plus loads/stores and loop overhead -
  call it ~50 instructions in plain code.
- A 1-megapixel image is then ~5e7 instructions -> tens of milliseconds, *if* every
  instruction is useful.

But it usually isn't, because of the **memory wall**: a multiply takes ~1 clock, a
load that misses the cache and goes to DRAM takes *hundreds*. The naive loop below
re-reads each pixel up to nine times, so it spends most of its life waiting on memory,
not multiplying. Keep that picture - it is the thread through the whole workshop.

### The scoreboard: how far from the limit?

We will keep one picture - a **roofline** - showing two ceilings the hardware cannot
beat: how fast it can compute, and how fast it can move data. Every implementation is
a dot under those ceilings. The gap is the performance still on the table.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

## Measuring honestly

Before optimising anything, you have to *measure* anything - and naive timing lies.
Two habits, used identically for every rung from here on (they live in
`fpga_conv/bench.py` so the method never changes):

- **Warm up.** The first run pays one-time costs that are not the computation: imports,
  the interpreter warming up, allocating scratch buffers, cold caches (and, later, the
  FPGA allocating its DMA buffers). Throw the first runs away.
- **Average, and look at the spread.** A single run is noisy - the OS schedules other
  work mid-measurement. Take many runs, report the *best* (the cleanest view of what
  the path sustains) and the spread, so the noise is visible instead of hidden.

`run_rung()` does exactly this and prints the cold-vs-warm gap so you can see why it
matters.

In [ ]:
# Pure Python: a triple loop over pixels and the 3x3 window. No NumPy - this is the
# thing you must NOT do, and the reason for everything that follows. Small image on
# purpose; even 64x64 takes a noticeable moment.
img = sample_gray(64)
out, t = run_rung('pyloop', img, 'edges', repeats=5, scoreboard=sb)

Look at the printout: the **cold first run** vs the **warm best**, and the median with
its spread. That difference is precisely what warmup and averaging are protecting you
from. Now the same numbers as a picture - the per-run times, with the cold run marked:

In [ ]:
bk_t = time_backend(__import__('fpga_conv').get_backend('pyloop'), img, 'edges', repeats=12)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, len(bk_t.samples_ms) + 1), bk_t.samples_ms, '-o', label='timed runs')
ax.axhline(bk_t.best_ms, color='green', ls='--', label=f'best {bk_t.best_ms:.2f} ms')
ax.set_xlabel('run #'); ax.set_ylabel('latency (ms)')
ax.set_title('Why we take the best of many runs'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Our first dot on the roofline

One implementation, measured. It sits far, *far* below both ceilings - the interpreter
is the bottleneck, not the hardware. Everything from here is closing that gap.

In [ ]:
sb.roofline(); plt.show()

**Next:** the single biggest, cheapest win - stop interpreting the inner loop. Hand the
nine-tap arithmetic to NumPy, which runs it in compiled C over the whole array at once.
-> `02_numpy.ipynb`.